# Earnings Event Engine Demo

This notebook is the canonical offline demo for owner review, walkthroughs, and future repo orientation. It uses the frozen research engine exactly as audited and runs the deterministic fixture end to end without network access.


In [1]:
from __future__ import annotations

import io
import shutil
import sys
import tempfile
from contextlib import redirect_stdout
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "scripts" / "run_full_research_pipeline.py").exists():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


def capture_stdout(func, *args, **kwargs):
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        value = func(*args, **kwargs)
    return value, buffer.getvalue()


def display_compact(df: pd.DataFrame, columns: list[str] | None = None, digits: int = 4):
    view = df.copy()
    if columns is not None:
        view = view.loc[:, columns]
    numeric_columns = view.select_dtypes(include=["number"]).columns
    if len(numeric_columns) > 0:
        view.loc[:, numeric_columns] = view.loc[:, numeric_columns].round(digits)
    display(view)


ROOT = find_repo_root(Path.cwd().resolve())
SRC_PATH = ROOT / "src"
local_paths = [str(ROOT), str(SRC_PATH)]
sys.path = local_paths + [path for path in sys.path if path not in local_paths]
for module_name in ["analysis", "scripts", "strategies"]:
    sys.modules.pop(module_name, None)

FIXTURE_DIR = ROOT / "tests" / "fixtures" / "offline_pipeline"
OUTPUT_DIR = Path(tempfile.gettempdir()) / "earnings_event_engine_demo"
NOTEBOOK_PATH = ROOT / "notebooks" / "01_earnings_event_engine_demo.ipynb"

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

print(f"repo_root={ROOT}")
print(f"fixture_dir={FIXTURE_DIR}")
print(f"demo_output_dir={OUTPUT_DIR}")
print(f"notebook_path={NOTEBOOK_PATH}")


repo_root=C:\GitHub\earnings_event_engine
fixture_dir=C:\GitHub\earnings_event_engine\tests\fixtures\offline_pipeline
demo_output_dir=C:\Users\jrsan\AppData\Local\Temp\earnings_event_engine_demo
notebook_path=C:\GitHub\earnings_event_engine\notebooks\01_earnings_event_engine_demo.ipynb


## 1. What This Engine Does

- normalizes earnings events into a consistent event contract
- aligns AMC and BMO events to the correct daily-bar sessions
- validates price and option inputs instead of silently guessing
- constructs a feature artifact that keeps diagnostics and invalid rows visible
- derives event-time option-implied move from explicit call and put mids
- restricts required downstream research to `validated_features.parquet`
- produces descriptive earnings analysis, a simplified straddle research proxy, and conditional comparisons


## 2. Deterministic Demo Setup

The fixture below is local, deterministic test data. It is useful for demonstrating plumbing and guardrails, not for claiming trading efficacy.


In [2]:
fixture_events = pd.read_csv(FIXTURE_DIR / "earnings_events.csv", parse_dates=["earnings_date"])
fixture_options = pd.read_csv(FIXTURE_DIR / "options.csv", parse_dates=["event_date"])
fixture_prices = pd.read_csv(FIXTURE_DIR / "prices.csv", parse_dates=["date"])

display(
    pd.DataFrame(
        {
            "item": ["Repository root", "Fixture directory", "Demo output directory", "Network access required"],
            "value": [str(ROOT), str(FIXTURE_DIR), str(OUTPUT_DIR), "No"],
        }
    )
)

display(Markdown("**Fixture earnings events**"))
display_compact(
    fixture_events.assign(earnings_date=fixture_events["earnings_date"].dt.strftime("%Y-%m-%d %H:%M")),
    ["ticker", "earnings_date", "announcement_timing"],
)

display(Markdown("**Fixture option mids used for the event-time implied-move contract**"))
display_compact(
    fixture_options.sort_values(["ticker", "option_type"]),
    ["ticker", "event_date", "option_type", "strike", "underlying_price", "bid", "ask", "mid"],
)

display(Markdown("**Fixture price panel coverage**"))
display_compact(
    fixture_prices.groupby("ticker", dropna=False).agg(price_rows=("date", "size"), first_date=("date", "min"), last_date=("date", "max")).reset_index(),
)


,item,value
0,Repository root,C:\GitHub\earnings_event_engine
1,Fixture directory,C:\GitHub\earnings_event_engine\tests\fixtures...
2,Demo output directory,C:\Users\jrsan\AppData\Local\Temp\earnings_eve...
3,Network access required,No


**Fixture earnings events**

,ticker,earnings_date,announcement_timing
0,AAPL,2025-01-30 16:05,AMC
1,MSFT,2025-02-05 08:00,BMO


**Fixture option mids used for the event-time implied-move contract**

,ticker,event_date,option_type,strike,underlying_price,bid,ask,mid
0,AAPL,2025-01-30,call,104.5,104.5,4.7,4.9,4.8
1,AAPL,2025-01-30,put,104.5,104.5,4.6,4.8,4.7
2,MSFT,2025-02-05,call,210.0,210.0,5.1,5.3,5.2
3,MSFT,2025-02-05,put,210.0,210.0,5.0,5.2,5.1


**Fixture price panel coverage**

,ticker,price_rows,first_date,last_date
0,AAPL,13,2025-01-15,2025-01-31
1,MSFT,13,2025-01-21,2025-02-06


## 3. Run the Canonical Pipeline

This notebook calls the audited orchestrator directly rather than copying pipeline logic into notebook cells.


In [3]:
from scripts.run_full_research_pipeline import print_run_summary, run_full_research_pipeline

report, pipeline_stdout = capture_stdout(
    run_full_research_pipeline,
    output_dir=OUTPUT_DIR,
    fixture_dir=FIXTURE_DIR,
    tickers=None,
    start_date="2024-01-01",
    end_date="2026-12-31",
    max_events=10,
    run_visuals=False,
)

summary_stdout = io.StringIO()
with redirect_stdout(summary_stdout):
    print_run_summary(report)

if report.status != "SUCCESS" or report.pipeline_artifacts is None:
    raise RuntimeError(f"Canonical pipeline failed: {report.failure_detail}")

pipeline_summary_lines = [
    line for line in (pipeline_stdout + "\n" + summary_stdout.getvalue()).splitlines()
    if line.startswith("===") or line.startswith("[SUCCESS]") or line.startswith("FINAL STATUS") or line.startswith("-")
]
print("\n".join(pipeline_summary_lines))

stage_summary = pd.DataFrame(
    [
        {
            "stage": stage.name,
            "required": stage.required,
            "status": stage.status,
            "detail": stage.detail,
            "elapsed_seconds": round(stage.elapsed_seconds, 2),
        }
        for stage in report.stage_results
    ]
)
display(stage_summary)

validated_features_path = report.pipeline_artifacts.validated_features_path
assert validated_features_path is not None
print(f"validated_features_path={validated_features_path}")


=== Full Research Pipeline ===
=== STAGE: event_pipeline ===
=== Diagnostics summary ===
=== Final dataset sample ===
=== Failure counts ===
- C:\Users\jrsan\AppData\Local\Temp\earnings_event_engine_demo\events.parquet
- C:\Users\jrsan\AppData\Local\Temp\earnings_event_engine_demo\clean_events.parquet
- C:\Users\jrsan\AppData\Local\Temp\earnings_event_engine_demo\features.parquet
- C:\Users\jrsan\AppData\Local\Temp\earnings_event_engine_demo\diagnostics.parquet
[SUCCESS] event_pipeline: raw_events=2 clean_events=2 feature_rows=2 validated_feature_rows=2
=== STAGE: earnings_analysis ===
=== Earnings Event Analysis ===
--- Basic Statistics ---
--- Distribution: realized_gap_move ---
--- Distribution: surprise_vs_implied ---
--- By Ticker (top 20 by count) ---
--- By Announcement Timing ---
[SUCCESS] earnings_analysis: by_ticker_rows=2 by_timing_rows=2
=== STAGE: strategy_simulation ===
- loaded_rows: 2
- option_eligible_rows: 2
- avg_pnl: -0.056950
- avg_cost: 0.069978
- avg_payoff: 0.01

,stage,required,status,detail,elapsed_seconds
0,event_pipeline,True,SUCCESS,raw_events=2 clean_events=2 feature_rows=2 val...,2.62
1,earnings_analysis,True,SUCCESS,by_ticker_rows=2 by_timing_rows=2,0.12
2,strategy_simulation,True,SUCCESS,simulation_rows=2 ticker_breakdown_rows=2,0.03
3,conditional_analysis,True,SUCCESS,group_rows=2 timing_rows=2,0.05
4,visualizations,False,SKIPPED,Optional stage disabled; canonical run exclude...,0.00


validated_features_path=C:\Users\jrsan\AppData\Local\Temp\earnings_event_engine_demo\validated_features.parquet


## 4. Inspect Event Artifacts

The canonical run emits raw normalized events, validated clean events, a feature audit artifact that may still contain invalid rows, a validated-only downstream artifact, and diagnostics.


In [4]:
events = pd.read_parquet(OUTPUT_DIR / "events.parquet")
clean_events = pd.read_parquet(OUTPUT_DIR / "clean_events.parquet")
features = pd.read_parquet(OUTPUT_DIR / "features.parquet")
validated_features = pd.read_parquet(validated_features_path)
diagnostics = pd.read_parquet(OUTPUT_DIR / "diagnostics.parquet")

artifact_summary = pd.DataFrame(
    {
        "artifact": ["events.parquet", "clean_events.parquet", "features.parquet", "validated_features.parquet", "diagnostics.parquet"],
        "rows": [len(events), len(clean_events), len(features), len(validated_features), len(diagnostics)],
        "path": [
            str(OUTPUT_DIR / "events.parquet"),
            str(OUTPUT_DIR / "clean_events.parquet"),
            str(OUTPUT_DIR / "features.parquet"),
            str(validated_features_path),
            str(OUTPUT_DIR / "diagnostics.parquet"),
        ],
    }
)
display(artifact_summary)

display(Markdown("**events.parquet**"))
display_compact(events, ["event_id", "ticker", "earnings_date", "event_date", "announcement_timing", "mapped_session"])

display(Markdown("**clean_events.parquet**"))
display_compact(clean_events, ["event_id", "ticker", "announcement_timing", "pre_close", "post_open", "post_close", "price_alignment_valid"])

display(Markdown("**features.parquet**"))
display_compact(features, ["event_id", "ticker", "announcement_timing", "option_snapshot_available", "implied_move_available", "valid_event", "invalid_event", "failure_reasons"])

display(Markdown("**validated_features.parquet**"))
display_compact(validated_features, ["event_id", "ticker", "announcement_timing", "implied_move", "realized_gap_move", "surprise_vs_implied", "valid_event"])

display(Markdown("**selected diagnostics**"))
diagnostics_view = diagnostics.loc[diagnostics["category"].isin(["overview", "coverage"]) | diagnostics["count"].gt(0)].copy()
display_compact(diagnostics_view)


,artifact,rows,path
0,events.parquet,2,C:\Users\jrsan\AppData\Local\Temp\earnings_eve...
1,clean_events.parquet,2,C:\Users\jrsan\AppData\Local\Temp\earnings_eve...
2,features.parquet,2,C:\Users\jrsan\AppData\Local\Temp\earnings_eve...
3,validated_features.parquet,2,C:\Users\jrsan\AppData\Local\Temp\earnings_eve...
4,diagnostics.parquet,20,C:\Users\jrsan\AppData\Local\Temp\earnings_eve...


**events.parquet**

,event_id,ticker,earnings_date,event_date,announcement_timing,mapped_session
0,1,AAPL,2025-01-30 16:05:00,2025-01-30,AMC,next_day
1,2,MSFT,2025-02-05 08:00:00,2025-02-05,BMO,same_day


**clean_events.parquet**

,event_id,ticker,announcement_timing,pre_close,post_open,post_close,price_alignment_valid
0,1,AAPL,AMC,106.0,108.4,107.8,True
1,2,MSFT,BMO,205.0,205.7,206.2,True


**features.parquet**

,event_id,ticker,announcement_timing,option_snapshot_available,implied_move_available,valid_event,invalid_event,failure_reasons
0,1,AAPL,AMC,True,True,True,False,
1,2,MSFT,BMO,True,True,True,False,


**validated_features.parquet**

,event_id,ticker,announcement_timing,implied_move,realized_gap_move,surprise_vs_implied,valid_event
0,1,AAPL,AMC,0.0909,0.0226,-0.0683,True
1,2,MSFT,BMO,0.0490,0.0034,-0.0456,True


**selected diagnostics**

,category,metric,count,pct_of_events
0,overview,total_events,2,1.0
1,overview,valid_events,2,1.0
2,overview,invalid_events,0,0.0
18,coverage,pct_missing_options,0,0.0
19,coverage,pct_missing_timing,0,0.0


## 5. Demonstrate Validated-Only Research

A key integrity feature of the repo is that required downstream research consumes `validated_features.parquet`, not every row in `features.parquet`.


In [5]:
raw_feature_rows = len(features)
validated_feature_rows = len(validated_features)
invalid_row_count = int(features["invalid_event"].astype("boolean").fillna(False).sum()) if "invalid_event" in features.columns else 0

validated_summary = pd.DataFrame(
    {
        "metric": ["raw_feature_rows", "validated_feature_rows", "invalid_row_count", "clean_event_rows"],
        "value": [raw_feature_rows, validated_feature_rows, invalid_row_count, len(clean_events)],
    }
)
display(validated_summary)

validated_event_ids = set(validated_features["event_id"].tolist())
clean_event_ids = set(clean_events["event_id"].tolist())
assert validated_event_ids == clean_event_ids

downstream_contract = pd.DataFrame(
    {
        "downstream_stage": ["earnings_analysis", "strategy_simulation", "conditional_analysis"],
        "actual_input_path": [str(validated_features_path), str(validated_features_path), "strategy_results derived from validated_features.parquet"],
    }
)
display(downstream_contract)


,metric,value
0,raw_feature_rows,2
1,validated_feature_rows,2
2,invalid_row_count,0
3,clean_event_rows,2


,downstream_stage,actual_input_path
0,earnings_analysis,C:\Users\jrsan\AppData\Local\Temp\earnings_eve...
1,strategy_simulation,C:\Users\jrsan\AppData\Local\Temp\earnings_eve...
2,conditional_analysis,strategy_results derived from validated_featur...


## 6. Option-Implied Move Contract

The real feature artifact retains both the option-derived implied move and the separate HV20 expected-move proxy. The option-derived value comes from event-time call and put mids and should not be confused with the volatility proxy.


In [6]:
option_legs = (
    fixture_options.pivot_table(
        index=["ticker", "event_date", "underlying_price", "strike"],
        columns="option_type",
        values="mid",
        aggfunc="first",
    )
    .reset_index()
    .rename(columns={"call": "fixture_call_mid", "put": "fixture_put_mid"})
)

contract_demo = validated_features.merge(option_legs, on=["ticker", "event_date"], how="left")
contract_demo["fixture_implied_move_check"] = (
    contract_demo["fixture_call_mid"] + contract_demo["fixture_put_mid"]
) / contract_demo["underlying_price"]
contract_demo["check_diff"] = contract_demo["implied_move"] - contract_demo["fixture_implied_move_check"]
contract_demo["implied_minus_hv_proxy"] = contract_demo["implied_move"] - contract_demo["hv20_expected_move_proxy"]

contract_columns = [
    "ticker",
    "event_date",
    "underlying_price",
    "fixture_call_mid",
    "fixture_put_mid",
    "implied_move",
    "fixture_implied_move_check",
    "check_diff",
    "implied_move_source",
    "hv20_expected_move_proxy",
    "implied_minus_hv_proxy",
]
display_compact(contract_demo, contract_columns, digits=6)

display(
    contract_demo.loc[:, ["ticker", "hv20_expected_move_proxy", "implied_move", "implied_minus_hv_proxy"]]
    .style.format(
        {
            "hv20_expected_move_proxy": "{:.4%}",
            "implied_move": "{:.4%}",
            "implied_minus_hv_proxy": "{:.4%}",
        }
    )
    .bar(subset=["hv20_expected_move_proxy", "implied_move"], color="#cfe8ff")
    .set_caption("Option-implied move and HV proxy are intentionally separate values.")
)


,ticker,event_date,underlying_price,fixture_call_mid,fixture_put_mid,implied_move,fixture_implied_move_check,check_diff,implied_move_source,hv20_expected_move_proxy,implied_minus_hv_proxy
0,AAPL,2025-01-30,104.5,4.8,4.7,0.090909,0.090909,0.0,event_option_straddle_mid_over_spot,0.003187,0.087722
1,MSFT,2025-02-05,210.0,5.2,5.1,0.049048,0.049048,-0.0,event_option_straddle_mid_over_spot,0.010507,0.038540


,ticker,hv20_expected_move_proxy,implied_move,implied_minus_hv_proxy
0,AAPL,0.3187%,9.0909%,8.7722%
1,MSFT,1.0507%,4.9048%,3.8540%


## 7. Realized vs Implied Earnings Analysis

This section uses the actual earnings-analysis stage on the validated-only artifact. With a two-event fixture, the output is descriptive plumbing, not evidence of predictive alpha.


In [7]:
from scripts.run_full_research_pipeline import run_analysis_stage

(analysis_outputs, analysis_detail), analysis_stdout = capture_stdout(run_analysis_stage, validated_features_path)
print(analysis_detail)

basic_statistics = pd.DataFrame(
    {
        "metric": list(analysis_outputs["basic_statistics"].keys()),
        "value": list(analysis_outputs["basic_statistics"].values()),
    }
)
display(basic_statistics)

display(Markdown("**By ticker**"))
display(analysis_outputs["by_ticker"].reset_index().round(6))

display(Markdown("**By announcement timing**"))
display(analysis_outputs["by_timing"].reset_index().round(6))


by_ticker_rows=2 by_timing_rows=2


,metric,value
0,event_count,2.000000
1,option_implied_event_count,2.000000
2,mean_realized_move,0.013028
3,mean_implied_move,0.069978
4,mean_surprise,-0.056950
5,pct_overpriced,100.000000
6,pct_underpriced,0.000000


**By ticker**

,ticker,event_count,option_implied_event_count,avg_realized_move,avg_implied_move,avg_surprise
0,AAPL,1,1,0.022642,0.090909,-0.068268
1,MSFT,1,1,0.003415,0.049048,-0.045633


**By announcement timing**

,announcement_timing,event_count,option_implied_event_count,avg_realized_move,avg_implied_move,avg_surprise
0,AMC,1,1,0.022642,0.090909,-0.068268
1,BMO,1,1,0.003415,0.049048,-0.045633


## 8. Simplified Straddle Research Proxy

- event-time option-implied move is used as the normalized cost proxy
- this is not observed historical contract-level entry and exit P&L
- the fixture demonstrates research plumbing, not profitability


In [8]:
from strategies.straddle_simulator import run_straddle_analysis

(strategy_results, strategy_summary, ticker_breakdown), strategy_stdout = capture_stdout(
    run_straddle_analysis,
    validated_features_path,
)

display(pd.DataFrame([strategy_summary]).round(6))
display_compact(
    strategy_results,
    ["event_id", "ticker", "announcement_timing", "implied_move", "realized_gap_move", "cost", "payoff", "pnl"],
    digits=6,
)
display(
    ticker_breakdown.reset_index()
    .round(6)
    .style.bar(subset=["avg_pnl"], color="#ffd9cc")
    .set_caption("Ticker-level proxy PnL remains descriptive only in this fixture.")
)


,avg_pnl,avg_cost,avg_payoff,win_rate,median_pnl,max_gain,max_loss
0,-0.05695,0.069978,0.013028,0.0,-0.05695,-0.045633,-0.068268


,event_id,ticker,announcement_timing,implied_move,realized_gap_move,cost,payoff,pnl
0,1,AAPL,AMC,0.090909,0.022642,0.090909,0.022642,-0.068268
1,2,MSFT,BMO,0.049048,0.003415,0.049048,0.003415,-0.045633


,ticker,avg_pnl,win_rate,median_pnl,max_gain,max_loss,n_events
0,MSFT,-0.045633,0.000000,-0.045633,-0.045633,-0.045633,1
1,AAPL,-0.068268,0.000000,-0.068268,-0.068268,-0.068268,1


## 9. Conditional Analysis

The deterministic fixture keeps these outputs mechanically useful for demo purposes, but the sample is too small for statistical interpretation.


In [9]:
from analysis.conditional_analysis import run_conditional_analysis

conditional_outputs, conditionals_stdout = capture_stdout(run_conditional_analysis, strategy_results)

for name in ["group_comparison", "timing_comparison", "macro_regime_comparison"]:
    if name in conditional_outputs:
        display(Markdown(f"**{name.replace('_', ' ').title()}**"))
        display(conditional_outputs[name].round(6))


**Group Comparison**

,group,event_count,avg_pnl,win_rate
0,high_iv,1,-0.068268,0.0
1,low_iv,1,-0.045633,0.0


**Timing Comparison**

,timing,event_count,avg_realized_gap_move,avg_pnl
0,AMC,1,0.022642,-0.068268
1,BMO,1,0.003415,-0.045633


**Macro Regime Comparison**

,macro_regime,event_count,avg_pnl,win_rate
0,unknown,2,-0.05695,0.0


## 10. Integrity and Guardrails

The table below summarizes the safeguards that frame the current scope. Some are demonstrated directly in this notebook, while others are maintained by the frozen engine and validated in the regression suite.


In [10]:
guardrails = pd.DataFrame(
    {
        "guardrail": [
            "Zero-event failure",
            "Retrieval failure distinction",
            "AMC/BMO session alignment",
            "Ambiguous timing rejection",
            "Historical IV leakage containment",
            "Event-time option-implied move requirement",
            "Validated-only downstream research",
            "Truthful orchestrator failure propagation",
        ],
        "demo_evidence": [
            "Engine guardrail preserved from audited test suite.",
            "Engine guardrail preserved from audited test suite.",
            "Visible above in normalized timing plus pre/post session alignment.",
            "Engine guardrail preserved from audited timing tests.",
            "README caveat plus audited IV containment tests preserve this boundary.",
            "Visible above through explicit call and put mids and implied_move_source.",
            "Visible above through validated_features.parquet row counts and downstream inputs.",
            "Visible above in the canonical run stage summary and status reporting.",
        ],
    }
)
display(guardrails)


,guardrail,demo_evidence
0,Zero-event failure,Engine guardrail preserved from audited test s...
1,Retrieval failure distinction,Engine guardrail preserved from audited test s...
2,AMC/BMO session alignment,Visible above in normalized timing plus pre/po...
3,Ambiguous timing rejection,Engine guardrail preserved from audited timing...
4,Historical IV leakage containment,README caveat plus audited IV containment test...
5,Event-time option-implied move requirement,Visible above through explicit call and put mi...
6,Validated-only downstream research,Visible above through validated_features.parqu...
7,Truthful orchestrator failure propagation,Visible above in the canonical run stage summa...


## 11. What Live Research Would Require

- reliable live research data availability
- real event-time historical option snapshots for historical options studies
- larger samples than this deterministic two-event fixture
- explicit transaction-cost and execution assumptions
- proper statistical validation before drawing research conclusions


In [11]:
assert report.status == "SUCCESS"
assert (OUTPUT_DIR / "events.parquet").exists()
assert (OUTPUT_DIR / "clean_events.parquet").exists()
assert (OUTPUT_DIR / "features.parquet").exists()
assert validated_features_path.exists()
assert len(features) >= len(validated_features) > 0
assert set(validated_features["event_id"]) == set(clean_events["event_id"])
assert validated_features["implied_move_available"].astype("boolean").fillna(False).all()
assert validated_features["implied_move_source"].eq("event_option_straddle_mid_over_spot").all()
print("Notebook validation checks passed.")
print("Matplotlib is not installed in this environment, so the demo uses concise tables and HTML bar styling instead of standalone charts.")


Notebook validation checks passed.
Matplotlib is not installed in this environment, so the demo uses concise tables and HTML bar styling instead of standalone charts.
